# 02 Kaggle Train (Single Run)

Runs one supervised training job for a single `(CATEGORY, RATIO)` pair, logs to W&B, and uploads checkpoints to a Kaggle Dataset version.

In [ ]:
# Parameters (edit before running)
GITHUB_REPO = "https://github.com/ManuelGamal/Self-Supervised-Industrial-Defect-Detection-System"
GITHUB_BRANCH = "main"

WANDB_ENTITY = "manuelaziz27-ain-shams-university"
WANDB_PROJECT = "defect-detection-supervised"

CATEGORY = "bottle"
RATIO = 100  # one of {100, 50, 10}
EPOCHS = 50
BATCH_SIZE = 32

# Dummy placeholders - replace with your actual paths/slugs
MVTEC_INPUT_DIR = "/kaggle/input/YOUR_MVTEC_DATASET/mvtec-ad"
CKPT_DATASET_SLUG = "YOUR_USERNAME/YOUR_CHECKPOINT_DATASET"

WORKDIR = "/kaggle/working/ssidds"
UPLOAD_DIR = "/kaggle/working/checkpoint_upload"

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

def get_secret(name):
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    return client.get_secret(name)

os.environ["WANDB_API_KEY"] = get_secret("WANDB_API_KEY")
os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")

if Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)

run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, WORKDIR])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=WORKDIR)
run(["wandb", "login", os.environ["WANDB_API_KEY"]])

assert Path(MVTEC_INPUT_DIR).exists(), f"MVTec path not found: {MVTEC_INPUT_DIR}"

In [ ]:
train_cmd = [
    sys.executable,
    "-m",
    "src.training.train",
    f"data.root={MVTEC_INPUT_DIR}",
    "data.splits_dir=data/splits",
    f"data.category={CATEGORY}",
    f"data.ratio={RATIO}",
    f"training.epochs={EPOCHS}",
    f"training.batch_size={BATCH_SIZE}",
    f"wandb.entity={WANDB_ENTITY}",
    f"wandb.project={WANDB_PROJECT}",
    "resume.auto=true",
]

run(train_cmd, cwd=WORKDIR)

In [ ]:
upload_dir = Path(UPLOAD_DIR)
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True, exist_ok=True)

summary_path = Path(WORKDIR) / "results" / "training" / CATEGORY / f"ratio{RATIO}" / "run_summary.json"
assert summary_path.exists(), f"Missing summary file: {summary_path}"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
best_model = Path(summary.get("best_model_path", ""))

if best_model.exists():
    shutil.copy2(best_model, upload_dir / f"{CATEGORY}_r{RATIO}_best.ckpt")

last_ckpt = Path(WORKDIR) / "checkpoints" / "supervised" / CATEGORY / f"ratio{RATIO}" / "last.ckpt"
if last_ckpt.exists():
    shutil.copy2(last_ckpt, upload_dir / f"{CATEGORY}_r{RATIO}_last.ckpt")

shutil.copy2(summary_path, upload_dir / f"{CATEGORY}_r{RATIO}_summary.json")

metadata = {
    "title": "Defect Detection Supervised Checkpoints",
    "id": CKPT_DATASET_SLUG,
    "licenses": [{"name": "CC0-1.0"}],
}
(upload_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

version_note = f"{CATEGORY} ratio {RATIO} supervised checkpoint"
run(["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", version_note, "-r", "zip"])

print("Uploaded files:")
for p in sorted(upload_dir.glob("*")):
    print(" -", p.name)